# 03. Neo4j GraphRAG — "Attention is All You Need"

`02_graph_build.ipynb`에서 구축한 지식 그래프를 기반으로  
**Neo4j GraphRAG**를 구현하고 논문 질의응답 및 요약 예제를 실습합니다.

## GraphRAG 파이프라인

```
사용자 질문
     │
     ▼
┌─────────────────────────────┐
│  Retriever (검색)           │
│  ① VectorRetriever         │ ← 의미 기반 벡터 검색
│  ② Text2CypherRetriever    │ ← 자연어 → Cypher 변환
└─────────────────────────────┘
     │ 관련 컨텍스트
     ▼
┌─────────────────────────────┐
│  OpenAI GPT-4o (LLM)       │
│  컨텍스트 + 질문 → 응답     │
└─────────────────────────────┘
     │
     ▼
  최종 응답
```

## 참고 자료
- `Neo4j_GenAI로_Retriever기반_GraphRAG_구현하기.ipynb`
- [neo4j-graphrag Python 패키지 문서](https://neo4j.com/docs/neo4j-graphrag-python/)

## Step 0. 패키지 설치

In [ ]:
!pip install neo4j-graphrag neo4j openai python-dotenv -q

## Step 1. 임포트 및 환경 설정

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

# neo4j-graphrag 패키지
from neo4j_graphrag.retrievers import VectorRetriever, Text2CypherRetriever
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG

# .env 로드
load_dotenv(".env", override=True)

# 환경변수
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_MODEL      = os.getenv("LLM_MODEL",      "gpt-4o")
EMBED_MODEL    = os.getenv("EMBEDDING_MODEL", "text-embedding-ada-002")

print(f"Neo4j URI      : {NEO4J_URI}")
print(f"LLM Model      : {LLM_MODEL}")
print(f"Embedding Model: {EMBED_MODEL}")

## Step 2. Neo4j 드라이버 및 OpenAI 초기화

In [ ]:
# Neo4j 드라이버
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("✅ Neo4j 연결 성공")

# OpenAI 임베딩 모델
embedder = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=OPENAI_API_KEY,
)
print(f"✅ OpenAI Embeddings 초기화: {EMBED_MODEL}")

# OpenAI LLM
llm = OpenAILLM(
    model_name=LLM_MODEL,
    model_params={"temperature": 0},
    api_key=OPENAI_API_KEY,
)
print(f"✅ OpenAI LLM 초기화: {LLM_MODEL}")

## Step 3. 그래프 스키마 정의 (Text2Cypher용)

`Text2CypherRetriever`가 올바른 Cypher를 생성하도록 그래프 스키마를 문자열로 정의합니다.

In [ ]:
NEO4J_SCHEMA = """
Node Labels and Properties:
- Paper(paper_id, title, authors, year, arxiv_id, venue)
- Section(id, number, title, level, content, page, embedding)
  * level: 1=top-level section, 2=subsection, 3=sub-subsection
  * number examples: '1', '2', '3', '3.1', '3.2.1'
- Table(id, number, caption, content, page, embedding)
- Figure(id, number, caption, description, page, embedding)
- Equation(id, number, formula, description, page, embedding)

Relationship Types:
- (Section)-[:IS_NEXT_TO]->(Section)       : sequential sections at same level
- (Section)-[:IS_BELONGING_TO]->(Section)  : child → parent hierarchy
- (Section)-[:CONTAINS]->(Section)         : parent → child hierarchy
- (Any)-[:PART_OF]->(Paper)               : node belongs to paper
- (Section)-[:REFERENCES]->(Table)         : section cites table
- (Section)-[:REFERENCES]->(Figure)        : section cites figure
- (Section)-[:INTRODUCES]->(Equation)      : section defines equation
- (Figure)-[:ILLUSTRATES]->(Section)       : figure explains section
- (Table)-[:SUPPORTS]->(Section)           : table supports section's claims
- (Table)-[:COMPARES]->(Section)           : table provides comparison data

Example IDs:
- Section IDs: 'section_1', 'section_3', 'section_3_2', 'section_3_2_1'
- Table IDs: 'table_1', 'table_2', 'table_3', 'table_4'
- Figure IDs: 'figure_1', 'figure_2', 'figure_3', 'figure_4', 'figure_5'
- Equation IDs: 'eq_1', 'eq_2', 'eq_3', 'eq_pe', 'eq_multihead'

Paper: paper_id='1706.03762', title='Attention Is All You Need'
"""

print("그래프 스키마 정의 완료")
print(NEO4J_SCHEMA)

## Step 4. VectorRetriever 설정

**VectorRetriever**: 질문을 임베딩하여 벡터 유사도 기반으로 관련 노드를 검색합니다.

> 📌 `02_graph_build.ipynb`에서 생성한 `section_embedding` 벡터 인덱스를 사용합니다.

In [ ]:
# ─── Section 벡터 검색기 ──────────────────────────────────────────
vector_retriever = VectorRetriever(
    driver=driver,
    index_name="section_embedding",     # 02_graph_build.ipynb에서 생성
    embedder=embedder,
    return_properties=["id", "number", "title", "content", "page", "level"],
    neo4j_database=NEO4J_DATABASE,
)

print("✅ VectorRetriever 초기화 완료")
print("   인덱스  : section_embedding")
print("   임베딩  : text-embedding-ada-002 (1536차원, cosine similarity)")

In [ ]:
# ─── VectorRetriever 검색 예제 1 ──────────────────────────────────
print("[예제 1] Multi-Head Attention 검색")
print("=" * 55)

query = "How does multi-head attention work?"
result = vector_retriever.search(query_text=query, top_k=3)

print(f"쿼리: {query}")
print(f"\n검색 결과 ({len(result.items)}개):")
for i, item in enumerate(result.items, 1):
    print(f"\n[{i}] {item.content[:200]}")
    print(f"    Score: {item.metadata.get('score', 'N/A'):.4f}" if item.metadata else "")

In [ ]:
# ─── VectorRetriever 검색 예제 2 ──────────────────────────────────
print("[예제 2] Positional Encoding 검색")
print("=" * 55)

query2 = "What is positional encoding and why is it needed?"
result2 = vector_retriever.search(query_text=query2, top_k=3)

print(f"쿼리: {query2}")
print(f"\n검색 결과 ({len(result2.items)}개):")
for i, item in enumerate(result2.items, 1):
    print(f"\n[{i}] {item.content[:200]}")

## Step 5. Text2CypherRetriever 설정

**Text2CypherRetriever**: 자연어 질문을 Cypher 쿼리로 변환하여 그래프를 탐색합니다.  
구조적 질문(계층 탐색, 관계 조회)에 특히 유용합니다.

In [ ]:
# Neo4j 5.x 문법에 맞는 Cypher를 생성하도록 커스텀 프롬프트 사용
T2C_CUSTOM_PROMPT = """Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:
{schema}

Examples:
{examples}

CRITICAL Cypher syntax rule (Neo4j 5.x):
When combining multiple relationship types, use the pipe `|` separator WITHOUT repeating the colon.
  Correct:   [:REFERENCES|INTRODUCES|CONTAINS]
  WRONG:     [:REFERENCES|:INTRODUCES|:CONTAINS]
Never write `|:` in relationship type expressions.

Input:
{query_text}

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:
"""

T2C_EXAMPLES = [
    "MATCH (s:Section)-[:REFERENCES|INTRODUCES|CONTAINS*1..2]-(n) RETURN s, n",
    "MATCH (s:Section)-[:IS_BELONGING_TO]->(parent:Section) RETURN s.title, parent.title",
    "MATCH (s:Section)-[:REFERENCES]->(t:Table) RETURN s.title, t.caption",
    "MATCH (s:Section)-[:IS_NEXT_TO*1..5]->(next:Section) WHERE s.number = '1' RETURN next.number, next.title",
]

t2c_retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=NEO4J_SCHEMA,
    neo4j_database=NEO4J_DATABASE,
    examples=T2C_EXAMPLES,
    custom_prompt=T2C_CUSTOM_PROMPT,
)

print("✅ Text2CypherRetriever 초기화 완료")

In [ ]:
# ─── Text2Cypher 예제 1: 섹션 구조 조회 ──────────────────────────
print("[예제 1] Attention 관련 섹션 조회")
print("=" * 55)

q = "What sections are related to the attention mechanism?"
t2c_result = t2c_retriever.search(query_text=q)

print(f"질문: {q}")
print(f"\n검색 결과:")
for i, item in enumerate(t2c_result.items, 1):
    print(f"  [{i}] {item.content}")

In [ ]:
# ─── Text2Cypher 예제 2: 표와 섹션의 관계 ────────────────────────
print("[예제 2] 비교 실험을 제공하는 표 조회")
print("=" * 55)

q2 = "Which tables provide comparison data for the Results section?"
t2c_result2 = t2c_retriever.search(query_text=q2)

print(f"질문: {q2}")
print(f"\n검색 결과:")
for i, item in enumerate(t2c_result2.items, 1):
    print(f"  [{i}] {item.content}")

In [ ]:
# ─── Text2Cypher 예제 3: 수식 도입 섹션 조회 ─────────────────────
print("[예제 3] 특정 섹션이 도입하는 수식 조회")
print("=" * 55)

q3 = "What equations does section 3.2.1 (Scaled Dot-Product Attention) introduce?"
t2c_result3 = t2c_retriever.search(query_text=q3)

print(f"질문: {q3}")
print(f"\n검색 결과:")
for i, item in enumerate(t2c_result3.items, 1):
    print(f"  [{i}] {item.content}")

In [ ]:
# ─── Text2Cypher 예제 4: 섹션 경로 탐색 ──────────────────────────
print("[예제 4] 섹션 3의 모든 하위 섹션 조회")
print("=" * 55)

q4 = "List all subsections that belong to section 3 (Model Architecture)."
t2c_result4 = t2c_retriever.search(query_text=q4)

print(f"질문: {q4}")
print(f"\n검색 결과:")
for i, item in enumerate(t2c_result4.items, 1):
    print(f"  [{i}] {item.content}")

## Step 6. GraphRAG 파이프라인 구성

**GraphRAG**: Retriever에서 검색된 컨텍스트를 LLM에 전달하여 최종 응답을 생성합니다.

In [ ]:
# VectorRetriever 기반 GraphRAG
rag = GraphRAG(
    retriever=vector_retriever,
    llm=llm,
)

print("✅ GraphRAG 파이프라인 초기화 완료")
print("   Retriever: VectorRetriever (section_embedding)")
print(f"   LLM      : {LLM_MODEL}")

## Step 7. 논문 요약 (Paper Summary) — 핵심 예제

> 📝 GraphRAG를 사용하여 "Attention Is All You Need" 논문의 핵심 내용을 요약합니다.

In [ ]:
# ─── 논문 전체 요약 ───────────────────────────────────────────────
print("=" * 60)
print("   논문 요약: Attention Is All You Need")
print("=" * 60)

summary_query = """
Please summarize the paper "Attention Is All You Need" (Transformer paper) 
in the following structured format (respond in Korean):

1. **핵심 아이디어**: 이 논문의 핵심 기여와 혁신점은 무엇인가?
2. **Transformer 모델 구조**: 인코더-디코더 구조, Multi-Head Attention, 
   Position-wise FFN, Positional Encoding을 포함하여 설명하라.
3. **Self-Attention의 장점**: 왜 RNN/CNN 대신 Self-Attention을 사용하는가?
4. **주요 실험 결과**: BLEU 점수 등 핵심 성능 지표를 포함하라.
5. **논문의 의의**: 이 논문이 NLP/AI 분야에 미친 영향은?
"""

summary_response = rag.search(
    query_text=summary_query,
    retriever_config={"top_k": 8},
)

print(summary_response.answer)

## Step 8. 다양한 RAG 질의응답 예제

In [ ]:
# ─── 예제 1: Transformer 아키텍처 설명 ────────────────────────────
print("[RAG 예제 1] Transformer 아키텍처")
print("-" * 55)

q1 = "Explain the encoder and decoder stack structure of the Transformer model in detail. (한국어로 답해주세요)"
r1 = rag.search(query_text=q1, retriever_config={"top_k": 5})
print(r1.answer)

In [ ]:
# ─── 예제 2: Scaled Dot-Product Attention 수식 설명 ───────────────
print("[RAG 예제 2] Scaled Dot-Product Attention 수식")
print("-" * 55)

q2 = """
Explain the Scaled Dot-Product Attention formula:
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

왜 sqrt(d_k)로 나누는가? Q, K, V는 각각 무엇을 의미하는가? (한국어로)
"""
r2 = rag.search(query_text=q2, retriever_config={"top_k": 4})
print(r2.answer)

In [ ]:
# ─── 예제 3: Self-Attention vs Recurrent 비교 (Table 1 활용) ──────
print("[RAG 예제 3] Self-Attention vs Recurrent 레이어 복잡도 비교")
print("-" * 55)

q3 = """
Compare the computational complexity of Self-Attention vs Recurrent layers.
What are the advantages of Self-Attention in terms of:
- Complexity per layer
- Sequential operations
- Maximum path length
(한국어로 답변, 표 데이터를 활용하여 구체적으로 설명)
"""
r3 = rag.search(query_text=q3, retriever_config={"top_k": 5})
print(r3.answer)

In [ ]:
# ─── 예제 4: BLEU 점수 결과 조회 (Table 2 활용) ───────────────────
print("[RAG 예제 4] 기계 번역 BLEU 점수 성능")
print("-" * 55)

q4 = """
What BLEU scores did the Transformer achieve on WMT 2014 machine translation tasks?
How does it compare to previous state-of-the-art models like GNMT and ConvS2S?
What is the training cost advantage? (한국어로)
"""
r4 = rag.search(query_text=q4, retriever_config={"top_k": 5})
print(r4.answer)

In [ ]:
# ─── 예제 5: Multi-Head Attention의 장점 ──────────────────────────
print("[RAG 예제 5] Multi-Head Attention의 장점")
print("-" * 55)

q5 = """
Why did the authors use Multi-Head Attention instead of single attention?
What are the benefits of having h=8 parallel attention heads with dimensions dk=dv=64?
(한국어로 설명)
"""
r5 = rag.search(query_text=q5, retriever_config={"top_k": 4})
print(r5.answer)

In [ ]:
# ─── 예제 6: Positional Encoding 설명 ────────────────────────────
print("[RAG 예제 6] Positional Encoding")
print("-" * 55)

q6 = """
Explain the Positional Encoding used in the Transformer:
- Why is positional information needed?
- How do the sine/cosine formulas work?
- What are the advantages over learned positional embeddings?
(한국어로 설명)
"""
r6 = rag.search(query_text=q6, retriever_config={"top_k": 4})
print(r6.answer)

## Step 9. Text2CypherRetriever 기반 GraphRAG

구조적 그래프 탐색 쿼리를 위한 Text2Cypher 기반 RAG 파이프라인입니다.

In [ ]:
# Text2Cypher 기반 GraphRAG
rag_t2c = GraphRAG(
    retriever=t2c_retriever,
    llm=llm,
)

print("✅ Text2Cypher GraphRAG 초기화 완료")

In [ ]:
# ─── Text2Cypher RAG 예제 1: 섹션 구조 네비게이션 ─────────────────
print("[Text2Cypher RAG 예제 1] 논문 전체 섹션 구조")
print("-" * 55)

tq1 = """
Show the complete section structure of the paper from section 1 to section 7,
including all subsections. Describe the overall organization of the paper.
(한국어로 구조를 설명)
"""
tr1 = rag_t2c.search(query_text=tq1)
print(tr1.answer)

In [ ]:
# ─── Text2Cypher RAG 예제 2: 그림과 섹션의 관계 ──────────────────
print("[Text2Cypher RAG 예제 2] 그림(Figure)과 관련 섹션")
print("-" * 55)

tq2 = """
List all figures in the paper and explain what sections they illustrate.
Describe what each figure shows about the Transformer model.
(한국어로)
"""
tr2 = rag_t2c.search(query_text=tq2)
print(tr2.answer)

In [ ]:
# ─── Text2Cypher RAG 예제 3: 수식 정보 조회 ──────────────────────
print("[Text2Cypher RAG 예제 3] 논문의 핵심 수식")
print("-" * 55)

tq3 = """
List all equations in the paper with their formulas and the sections that introduce them.
Explain the role of each equation in the Transformer model.
(한국어로)
"""
tr3 = rag_t2c.search(query_text=tq3)
print(tr3.answer)

## Step 10. 그래프 직접 탐색 쿼리 (Cypher)

Neo4j 드라이버로 직접 Cypher 쿼리를 실행하여 그래프 구조를 탐색합니다.

In [ ]:
# ─── 섹션 5 (Training)에서 시작하는 전체 경로 ─────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH p = (s:Section {number: '5'})-[:CONTAINS*1..2]->(sub:Section)
        RETURN s.title AS parent, sub.number AS sub_num, sub.title AS sub_title
        ORDER BY sub.number
    """).data()

print("=== 섹션 5 (Training) 하위 구조 ===")
for row in result:
    print(f"  {row['parent']} → {row['sub_num']}: {row['sub_title']}")

In [ ]:
# ─── 논문 순서대로 최상위 섹션 탐색 ──────────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH p = (start:Section)-[:IS_NEXT_TO*]->(end:Section)
        WHERE start.number = '1' AND start.level = 1
        AND NOT ()-[:IS_NEXT_TO]->(start)
        RETURN [s IN nodes(p) | s.number + '. ' + s.title] AS path
        ORDER BY length(p) DESC
        LIMIT 1
    """).data()

print("=== 논문 최상위 섹션 순서 ===")
if result:
    for section in result[0]['path']:
        print(f"  → {section}")
else:
    # fallback: 직접 조회
    result2 = session.run("""
        MATCH (s:Section) WHERE s.level = 1
        RETURN s.number AS num, s.title AS title
        ORDER BY toFloat(s.number)
    """).data()
    for row in result2:
        print(f"  {row['num']}. {row['title']}")

In [ ]:
# ─── 표(Table)와 그것이 지원하는 섹션 연결 확인 ───────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH (t:Table)-[r]->(s:Section)
        RETURN t.id AS table_id,
               t.number AS table_num,
               t.caption AS caption,
               type(r) AS rel_type,
               s.number AS section_num,
               s.title AS section_title
        ORDER BY t.number
    """).data()

print("=== 표(Table) ─ 섹션 연결 관계 ===")
for row in result:
    print(f"  Table {row['table_num']}: --{row['rel_type']}--> Section {row['section_num']} ({row['section_title']})")
    print(f"    캡션: {row['caption'][:70]}...")

## Step 11. 논문 심층 요약 (Extended Summary)

그래프의 다양한 노드(섹션 + 표 + 수식)를 활용한 심층 요약입니다.

In [ ]:
# 그래프에서 핵심 정보 직접 수집
with driver.session(database=NEO4J_DATABASE) as session:
    # 섹션 내용 수집
    sections = session.run("""
        MATCH (s:Section) WHERE s.level = 1
        RETURN s.number AS num, s.title AS title, s.content AS content
        ORDER BY toFloat(s.number)
    """).data()
    
    # 수식 수집
    equations = session.run("""
        MATCH (e:Equation)
        RETURN e.number AS num, e.formula AS formula, e.description AS desc
        ORDER BY e.number
    """).data()
    
    # 표 캡션 수집
    tables = session.run("""
        MATCH (t:Table)
        RETURN t.number AS num, t.caption AS caption, t.content AS content
        ORDER BY t.number
    """).data()

# 심층 요약용 컨텍스트 구성
context_parts = []
context_parts.append("=== PAPER SECTIONS ===")
for s in sections:
    context_parts.append(f"Section {s['num']}: {s['title']}\n{s['content'][:400] if s['content'] else ''}")

context_parts.append("\n=== KEY EQUATIONS ===")
for e in equations:
    context_parts.append(f"Eq({e['num']}): {e['formula']}\n{e['desc'][:200]}")

context_parts.append("\n=== TABLES ===")
for t in tables:
    context_parts.append(f"Table {t['num']}: {t['caption'][:150]}\n{t['content'][:300]}")

graph_context = "\n\n".join(context_parts)
print(f"컨텍스트 구성 완료: {len(graph_context):,} 문자")

In [ ]:
# 그래프 컨텍스트를 직접 사용한 심층 요약
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

deep_summary_prompt = f"""
아래는 "Attention Is All You Need" 논문의 핵심 내용입니다.
이 정보를 바탕으로 논문의 심층 요약을 한국어로 작성해주세요.

{graph_context[:6000]}

---
요약 형식:

# Attention Is All You Need — 심층 요약

## 1. 연구 배경 및 동기

## 2. 핵심 아이디어: Transformer 모델

## 3. 주요 구성 요소
### 3.1 인코더-디코더 구조
### 3.2 Multi-Head Attention
### 3.3 Position-wise FFN
### 3.4 Positional Encoding

## 4. 핵심 수식

## 5. 실험 결과

## 6. 논문의 의의와 영향
"""

response = openai_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "You are an expert AI researcher specializing in NLP and transformer models."},
        {"role": "user", "content": deep_summary_prompt},
    ],
    temperature=0,
)

print(response.choices[0].message.content)

## Step 12. 결과 요약

구현한 GraphRAG 시스템의 기능을 정리합니다.

In [ ]:
print("=" * 60)
print("   Transformer 논문 GraphRAG 구현 완료")
print("=" * 60)
print("""
구현된 기능:

① VectorRetriever
   - 의미 기반 벡터 검색 (cosine similarity)
   - 인덱스: section_embedding (text-embedding-ada-002)
   - 예제: Multi-Head Attention, Positional Encoding 검색

② Text2CypherRetriever
   - 자연어 → Cypher 쿼리 자동 변환
   - 그래프 구조 탐색 (섹션 계층, 관계 추적)
   - 예제: 섹션/표/그림/수식 구조 조회

③ GraphRAG
   - Retriever + GPT-4o 통합 파이프라인
   - 논문 요약, 구조 설명, 수식 해설, 성능 비교

④ 논문 심층 요약
   - 그래프 컨텍스트 기반 구조화된 요약
   - 수식 + 표 + 섹션 내용 통합
""")

# 드라이버 종료
driver.close()
print("✅ Neo4j 연결 종료")